# 02. Training Notebook

Build the detection model, create loaders, train Faster R-CNN, and save the checkpoint.

In [ ]:
import os
from pathlib import Path
from typing import Literal

import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt

print("PyTorch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
COCO_SYMLINK = Path("data/coco")


def find_kaggle_coco_root() -> Path | None:
    direct_candidates = [
        Path("/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017"),
        Path("/kaggle/input/coco-2017-dataset/coco2017"),
    ]
    for candidate in direct_candidates:
        if (candidate / "train2017").exists() and (candidate / "val2017").exists() and (candidate / "annotations").exists():
            return candidate

    if KAGGLE_INPUT_ROOT.exists():
        for candidate in sorted(KAGGLE_INPUT_ROOT.glob("**/coco2017")):
            if (candidate / "train2017").exists() and (candidate / "val2017").exists() and (candidate / "annotations").exists():
                return candidate

    return None


kaggle_coco_root = find_kaggle_coco_root()
if kaggle_coco_root is not None and not COCO_SYMLINK.exists():
    COCO_SYMLINK.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(kaggle_coco_root, COCO_SYMLINK, target_is_directory=True)
    print(f"Created symlink: {COCO_SYMLINK} -> {kaggle_coco_root}")
elif COCO_SYMLINK.exists():
    print(f"Using existing COCO path: {COCO_SYMLINK.resolve()}")
else:
    print("No Kaggle COCO dataset detected automatically. Make sure data/coco exists.")


In [ ]:
# Create the project directories used during setup, training, and evaluation.
for path in [
    Path("data"),
    Path("results"),
    Path("figures"),
    Path("figures/generated_samples"),
    Path("figures/prediction_visualizations"),
]:
    path.mkdir(parents=True, exist_ok=True)

print("Directory setup completed.")

### Kaggle Note

This notebook can run directly on Kaggle if the COCO 2017 dataset is attached as an input.  
The setup cell above creates a `data/coco` symlink automatically when it detects a Kaggle COCO input.


---
## 2. Load Exported Synthetic Dataset

This notebook trains the detector from the exported dataset created by `01_data_creation.ipynb`.


In [ ]:
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_V2_Weights, fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor


def build_model(task: str, pretrained: bool):
    """
    Build and return the main detection model.
    For this notebook we implement Faster R-CNN with a ResNet-50 FPN v2 backbone.
    """
    if task != "detection":
        raise NotImplementedError("Only detection is implemented in this notebook.")

    weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT if pretrained else None
    model = fasterrcnn_resnet50_fpn_v2(weights=weights, weights_backbone=None)

    num_classes = 2  # background + synthetic_shape
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


TASK      = "detection"   # or "segmentation"
PRETRAINED = True
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"

model = build_model(TASK, PRETRAINED).to(DEVICE)

---
## 6. Training Procedure

Required training details to report (§7):

| Detail | Value |
|--------|-------|
| Input image size | |
| Batch size | |
| Number of epochs | |
| Optimizer | |
| Learning rate | |
| Loss function | |
| Pretrained weights | |
| Hardware | |
| Approximate training time | |

In [ ]:
import json
import torchvision.transforms as T

EXPORTED_DATASET_ROOT = Path("data/synthetic_detection")
if not EXPORTED_DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Exported dataset not found at {EXPORTED_DATASET_ROOT}. Run 01_data_creation.ipynb first."
    )


def collate_fn(batch):
    return tuple(zip(*batch))


def get_transforms(split: str, use_augmentation: bool = False):
    transforms = []
    if split == "train" and use_augmentation:
        transforms.extend([
            T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.05, hue=0.02),
            T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.2)),
        ])
    transforms.append(T.ToTensor())
    return T.Compose(transforms)


class ExportedSyntheticDetectionDataset(Dataset):
    def __init__(self, root: str | Path, split: Literal["train", "val", "test"], transform=None):
        self.root = Path(root)
        self.split = split
        self.transform = transform or T.ToTensor()
        self.image_root = self.root / split / "images"
        self.records = json.loads((self.root / split / "annotations.json").read_text())

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx: int):
        record = self.records[idx]
        image = Image.open(self.image_root / record["file_name"]).convert("RGB")
        image = self.transform(image)

        boxes = record["boxes"]
        if boxes:
            boxes_tensor = torch.tensor(boxes, dtype=torch.float32)
            labels_tensor = torch.tensor(record["labels"], dtype=torch.long)
        else:
            boxes_tensor = torch.zeros((0, 4), dtype=torch.float32)
            labels_tensor = torch.zeros((0,), dtype=torch.long)

        target = {
            "boxes": boxes_tensor,
            "labels": labels_tensor,
            "image_id": record["image_id"],
            "is_positive": bool(record["is_positive"]),
        }
        return image, target


train_ds = ExportedSyntheticDetectionDataset(EXPORTED_DATASET_ROOT, "train", get_transforms("train"))
val_ds = ExportedSyntheticDetectionDataset(EXPORTED_DATASET_ROOT, "val", get_transforms("val"))

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)

print(f"Loaded exported dataset from {EXPORTED_DATASET_ROOT}")
print(f"Train samples: {len(train_ds)}  Val samples: {len(val_ds)}")


---
## 8. Evaluation Metrics

In [ ]:
# Evaluation helpers and exported test split

test_ds = ExportedSyntheticDetectionDataset(EXPORTED_DATASET_ROOT, "test", get_transforms("val"))
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)


def _to_box_tensor(boxes) -> torch.Tensor:
    if isinstance(boxes, torch.Tensor):
        return boxes.detach().cpu().to(torch.float32)
    if boxes is None:
        return torch.zeros((0, 4), dtype=torch.float32)
    return torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)


def _box_iou_matrix(boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
    if boxes1.numel() == 0 or boxes2.numel() == 0:
        return torch.zeros((boxes1.shape[0], boxes2.shape[0]), dtype=torch.float32)

    top_left = torch.maximum(boxes1[:, None, :2], boxes2[None, :, :2])
    bottom_right = torch.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])
    wh = (bottom_right - top_left).clamp(min=0)
    inter = wh[..., 0] * wh[..., 1]

    area1 = ((boxes1[:, 2] - boxes1[:, 0]).clamp(min=0) * (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0))
    area2 = ((boxes2[:, 2] - boxes2[:, 0]).clamp(min=0) * (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0))
    union = area1[:, None] + area2[None, :] - inter
    return torch.where(union > 0, inter / union, torch.zeros_like(inter))


def _match_boxes_greedy(pred_boxes: torch.Tensor, gt_boxes: torch.Tensor, iou_threshold: float):
    ious = _box_iou_matrix(pred_boxes, gt_boxes)
    matches = []

    while ious.numel() > 0:
        max_iou = float(ious.max().item())
        if max_iou < iou_threshold:
            break
        flat_idx = int(ious.argmax().item())
        pred_idx = flat_idx // ious.shape[1]
        gt_idx = flat_idx % ious.shape[1]
        matches.append((pred_idx, gt_idx, max_iou))
        ious[pred_idx, :] = -1.0
        ious[:, gt_idx] = -1.0

    return matches


def detection_metrics(predictions: list, targets: list, iou_threshold: float = 0.5) -> dict:
    tp = 0
    fp = 0
    fn = 0
    matched_ious = []

    for pred, tgt in zip(predictions, targets):
        pred_boxes = _to_box_tensor(pred.get("boxes"))
        gt_boxes = _to_box_tensor(tgt.get("boxes"))

        if "scores" in pred and len(pred["scores"]) == len(pred_boxes):
            order = torch.argsort(pred["scores"].detach().cpu(), descending=True)
            pred_boxes = pred_boxes[order]

        matches = _match_boxes_greedy(pred_boxes, gt_boxes, iou_threshold)
        tp += len(matches)
        fp += len(pred_boxes) - len(matches)
        fn += len(gt_boxes) - len(matches)
        matched_ious.extend(match_iou for _, _, match_iou in matches)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    mean_iou = float(sum(matched_ious) / len(matched_ious)) if matched_ious else 0.0

    return {
        "precision@0.5": precision,
        "recall@0.5": recall,
        "f1@0.5": f1,
        "mean_iou_matched": mean_iou,
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
    }


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


def _move_targets_to_device(targets, device):
    moved = []
    for target in targets:
        moved.append({
            key: value.to(device) if isinstance(value, torch.Tensor) else value
            for key, value in target.items()
        })
    return moved


def train_one_epoch(model, loader, optimizer, device) -> float:
    """Run one training epoch; return average loss."""
    model.train()
    total_loss = 0.0
    num_batches = 0

    for images, targets in loader:
        images = [image.to(device) for image in images]
        targets = _move_targets_to_device(targets, device)

        loss_dict = model(images, targets)
        loss = sum(loss_value for loss_value in loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item())
        num_batches += 1

    return total_loss / max(num_batches, 1)


def evaluate_val(model, loader, device) -> dict:
    """Evaluate on validation set; return metric dict. Do NOT use on test set for model selection."""
    model.eval()
    predictions, targets_all = [], []

    with torch.no_grad():
        for images, targets in loader:
            images_device = [image.to(device) for image in images]
            outputs = model(images_device)

            outputs_cpu = [
                {key: value.detach().cpu() if isinstance(value, torch.Tensor) else value for key, value in output.items()}
                for output in outputs
            ]
            targets_cpu = [
                {key: value.detach().cpu() if isinstance(value, torch.Tensor) else value for key, value in target.items()}
                for target in targets
            ]

            predictions.extend(outputs_cpu)
            targets_all.extend(targets_cpu)

    return detection_metrics(predictions, targets_all)


import time, json
history = []

for epoch in range(1, 11):
    t0 = time.time()
    train_loss  = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_metrics = evaluate_val(model, val_loader, DEVICE)
    elapsed     = time.time() - t0

    print(f"Epoch {epoch:02d}  loss={train_loss:.4f}  {val_metrics}  ({elapsed:.1f}s)")
    history.append({"epoch": epoch, "train_loss": train_loss, **val_metrics})

Path("results").mkdir(exist_ok=True)
Path("results/metrics.json").write_text(json.dumps(history, indent=2))

In [ ]:
MODEL_DIR = Path("results/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "fasterrcnn_resnet50_fpn_v2_synthetic.pth"
torch.save(model.state_dict(), MODEL_PATH)
print(f"Saved trained model to {MODEL_PATH}")
